- Aqui tentarei visualizar e analisar os times outliers na história recente da NFL, o que tornava eles "especiais", e se isso resultou em sucesso ou não.

In [3]:
import pandas as pd
import numpy as np
import nflreadpy as nfl
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# importação dos dados brutos 

ANO_ATUAL = 2026
ultimos_5_anos = list(range(ANO_ATUAL - 5, ANO_ATUAL))

df_jogos = nfl.load_schedules(ultimos_5_anos).to_pandas()
df_pstats = nfl.load_player_stats(ultimos_5_anos).to_pandas()

# separando a temporada regular (para achar as anomalias) e o SB (para verificar sucesso)
df_reg = df_jogos[df_jogos['game_type'] == 'REG'].copy()
df_sb = df_jogos[df_jogos['game_type'] == 'SB'].copy()

# agregando estatísticas por jogo e time

df_pstats['turnovers_cometidos'] = df_pstats['passing_interceptions'] + df_pstats['fumbles_lost_total']

# somando a produção por time dentro de cada jogo específico
df_game_team = df_pstats.groupby(['season', 'game_id', 'team']).agg(
    Pass_Yds=('passing_yards', 'sum'),
    Rush_Yds=('rushing_yards', 'sum'),
    Carries=('carries', 'sum'), # controle de relógio
    Turnovers=('turnovers_cometidos', 'sum'),
    Sacks_Suffered=('sacks_suffered', 'sum') # base do Pass Rush
).reset_index()

# cruzando mandante e visitante

df_games = df_reg[['season', 'game_id', 'home_team', 'away_team', 'home_score', 'away_score']].copy()

# trazendo status do Mandante
df_games = df_games.merge(df_game_team, left_on=['game_id', 'home_team'], right_on=['game_id', 'team'], how='left')
df_games.rename(columns={'Pass_Yds':'home_pass', 'Rush_Yds':'home_rush', 'Carries':'home_carries', 
                         'Turnovers':'home_to', 'Sacks_Suffered':'home_sacks'}, inplace=True)
df_games.drop(columns=['team'], inplace=True, errors='ignore')

# trazendo status do Visitante
df_games = df_games.merge(df_game_team, left_on=['game_id', 'away_team'], right_on=['game_id', 'team'], how='left')
df_games.rename(columns={'Pass_Yds':'away_pass', 'Rush_Yds':'away_rush', 'Carries':'away_carries', 
                         'Turnovers':'away_to', 'Sacks_Suffered':'away_sacks'}, inplace=True)
df_games.drop(columns=['team'], inplace=True, errors='ignore')

df_games.fillna(0, inplace=True)

# construindo o perfil do time

# visão do Mandante
df_home_persp = pd.DataFrame({
    'season': df_games['season'], 'team': df_games['home_team'],
    'PF': df_games['home_score'], 'PA': df_games['away_score'],
    'Off_Pass_Yds': df_games['home_pass'], 'Off_Rush_Yds': df_games['home_rush'],
    'Off_Carries': df_games['home_carries'], 'Off_Turnovers': df_games['home_to'],
    'Def_Sacks_Produced': df_games['away_sacks'], # sacks que a defesa APLICOU
    'Def_Turnovers_Forced': df_games['away_to']   # turnovers que a defesa ROUBOU
})

# visão do Visitante
df_away_persp = pd.DataFrame({
    'season': df_games['season'], 'team': df_games['away_team'],
    'PF': df_games['away_score'], 'PA': df_games['home_score'],
    'Off_Pass_Yds': df_games['away_pass'], 'Off_Rush_Yds': df_games['away_rush'],
    'Off_Carries': df_games['away_carries'], 'Off_Turnovers': df_games['away_to'],
    'Def_Sacks_Produced': df_games['home_sacks'],
    'Def_Turnovers_Forced': df_games['home_to']
})

# vgrupando a temporada inteira
df_season = pd.concat([df_home_persp, df_away_persp]).groupby(['season', 'team']).sum().reset_index()

In [5]:
# verificando o sucesso pelo superbowl

def status_superbowl(row):
    sb_ano = df_sb[df_sb['season'] == row['season']]
    if sb_ano.empty: return '-' 
    home, away = sb_ano.iloc[0]['home_team'], sb_ano.iloc[0]['away_team']
    vencedor = home if sb_ano.iloc[0]['home_score'] > sb_ano.iloc[0]['away_score'] else away
    if row['team'] == vencedor: return '🏆 Campeão'
    elif row['team'] in [home, away]: return '🥈 Vice'
    else: return '❌ Não Chegou'

df_season['Status_SB'] = df_season.apply(status_superbowl, axis=1)

# ISOLATION FOREST
# o modelo foca nos pilares que vencem jogos
features = ['PF', 'PA', 'Off_Pass_Yds', 'Off_Rush_Yds', 'Off_Carries', 
            'Off_Turnovers', 'Def_Sacks_Produced', 'Def_Turnovers_Forced']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_season[features])

iso = IsolationForest(contamination=0.10, random_state=42)
df_season['Anomalia'] = iso.fit_predict(X_scaled)
df_season['Score_Anomalia'] = iso.decision_function(X_scaled) 

- A coluna Anomalia indica se os times que lideraram cada ano era apenas o "melhor daquele ano" (1) ou se era uma "aberração estatística histórica" (-1).

In [6]:
# exibindo os outliers

colunas_off = ['team', 'PF', 'Off_Pass_Yds', 'Off_Rush_Yds', 'Off_Carries', 'Off_Turnovers', 'Status_SB', 'Anomalia']
colunas_def = ['team', 'PA', 'Def_Sacks_Produced', 'Def_Turnovers_Forced', 'Status_SB', 'Anomalia']

anos_unicos = sorted(df_season['season'].unique())

for ano in anos_unicos:
    df_ano = df_season[df_season['season'] == ano]
    
    # Pegando os 3 melhores ataques (Mais pontos) e 3 melhores defesas (Menos pontos) daquela temporada
    top_off_ano = df_ano.sort_values(by='PF', ascending=False).head(3)
    top_def_ano = df_ano.sort_values(by='PA', ascending=True).head(3)
    
    print(f"\n{'='*30} TEMPORADA {ano} {'='*30}")
    
    print("🔥 TOP 3 ATAQUES (Ordenados por Pontos Feitos)")
    print(top_off_ano[colunas_off].to_string(index=False))
    
    print("\n🧱 TOP 3 DEFESAS (Ordenadas por Pontos Sofridos)")
    print(top_def_ano[colunas_def].to_string(index=False))


============================== TEMPORADA 2021 ==============================
🔥 TOP 3 ATAQUES (Ordenados por Pontos Feitos)
team  PF  Off_Pass_Yds  Off_Rush_Yds  Off_Carries  Off_Turnovers    Status_SB  Anomalia
 DAL 530          4963          2119          473             20 ❌ Não Chegou         1
  TB 511          5383          1672          385             19 ❌ Não Chegou        -1
 BUF 483          4450          2209          461             22 ❌ Não Chegou         1

🧱 TOP 3 DEFESAS (Ordenadas por Pontos Sofridos)
team  PA  Def_Sacks_Produced  Def_Turnovers_Forced    Status_SB  Anomalia
 BUF 289                  42                    30 ❌ Não Chegou         1
  NE 303                  36                    30 ❌ Não Chegou         1
 DEN 322                  36                    19 ❌ Não Chegou         1

============================== TEMPORADA 2022 ==============================
🔥 TOP 3 ATAQUES (Ordenados por Pontos Feitos)
team  PF  Off_Pass_Yds  Off_Rush_Yds  Off_Carries  Off_

- Temporada 2021: A ilusão do jogo aéreo
    - O que aconteceu: O Tampa Bay Buccaneers foi a única aberração histórica do ano (-1). Eles lançaram para absurdas 5.383 jardas aéreas, mas o controle de relógio deles foi fraco (apenas 385 carries e 1.672 jardas terrestres).
    - Insight: Lembrando que a variável Pass_Forte (jardas aéreas) não apareceu no Top 15 do FP-Growth(regras de associação), o Isolation Forest confirmou por quê: um ataque aéreo anômalo, sem controle de relógio terrestre, não foi suficiente nem para chegar ao Super Bowl.

- Temporada 2022: O rolo compressor terrestre
    - O que aconteceu: O Philadelphia Eagles chegou ao Super Bowl (Vice) com o selo de Anomalia Ofensiva histórica(-1) por causa de suas 544 corridas e 2.509 jardas terrestres, tendo um bom sucesso.
    - Insight: Esse é o combo do fp-growth '[Controle_Relogio] -> [Rush_Forte]' agindo na prática. Eles literalmente esmagaram o cronômetro para chegar à grande final. Mas, o Kansas City Chiefs foi o campeão como o melhor ataque não-anômalo, equilibrando bem seu ótimo ataque aéreo ao seu razoável ataque terrestre.

- Temporada 2023: O Pass Rush dita as regras 
    - O que aconteceu: O Kansas City Chiefs levanta a taça construindo seu título puramente na defesa. Eles foram a 2ª melhor defesa do ano, aplicando impressionantes 57 Sacks.
    - Insight: Isso valida brutalmente a regra '[Pass_Rush_Elite]'. O time que consegue derreter o QB adversário pavimenta seu caminho nos playoffs. O Vice (SF) também estava no Top 3 defensivo com 48 sacks.
    - Nota triste: Baltimore foi uma anomalia defensiva histórica (-1) com 60 sacks e 31 posses roubadas, mas falhou. A defesa sozinha até ganha jogo, mas sem um ataque que aproveite esses turnovers, o time cai nos playoffs. Corroborando uma problemática bem conhecida pra quem assiste, que é os 'chokes' do QB Lamar Jackson.

- Temporada 2024: A Obra-Prima do modelo
    - O que aconteceu: O ano mais caótico. Os três ataques mais monstruosos da década (DET, BUF e BAL), todos eles sendo outliers históricos(-1). Baltimore correu incríveis 554 vezes. Detroit marcou 564 pontos. Buffalo com um equiíbrio muito alto. O que aconteceu com eles? Nenhum chegou ao Super Bowl. O Campeão foi o Philadelphia Eagles, coroado como a única Defesa Anômala (-1) do ano (41 sacks e 26 turnovers forçados).
    - Insight: Esse é o triunfo da defesa. Ataques históricos chocam o mundo na temporada regular (fazem muitos pontos e quebram recordes), mas nos playoffs, onde o jogo fica truncado, é a Defesa de Elite (como previu a regra de Lift alto do fp-growth) que levanta o troféu.

- Temporada 2025: A "Condição de Perfeição" comprovada
    - O que aconteceu: O Seattle Seahawks vence o Super Bowl de 2025. E olhando para as duas tabelas: eles são o 3º melhor Ataque do ano, e a melhor Defesa do ano, apesar de não serem outliers históricos em nenhum dos dois lados.
    - Insight: Lembrando daquela regra inicial '[Defesa_Elite, Ataque_Elite] -> [Vitoria]' que tinha Confiança de 1.00 e Lift de 2.01, O Seattle Seahawks de 2025 é a encarnação dessa regra. A sinergia perfeita entre dominar as trincheiras defensivas e controlar a bola no ataque, sendo um time de elite tanto na defesa quanto no ataque, garantiu o título sem que eles precisassem ser uma anomalia matemática extrema em nenhum dos dois.

- Resumidamente, os dois modelos, até agora, confirmaram a mesma verdade por caminhos matemáticos completamente diferentes. Um Super Bowl não se vence quebrando recordes de passes, mas sim combinando Pass Rush (Sacks) na Defesa, com Controle de Relógio (Carries) no Ataque. E, quando um time tenta quebrar a liga com um ataque anômalo focado apenas em pontuar insanamente (como a classe de 2024), a defesa sempre cobra a conta nos playoffs.